# 03 — every trade against the book in force when it printed

DuckDB's `ASOF JOIN` pairs each trade with the last known BBO. `gold.bbo_1s` is the book at the **end** of its second, so the quote in force for a trade at `exchange_ts` is the row for the *previous* second — the join is on `second + 1 s <= exchange_ts`. Joining on `second <= exchange_ts` would use a quote from up to a second in the trade's future, and reads as 77 % of Binance prints trading through the book. From the correct pairing: inside, at, or through the quote, and the effective spread paid.

In [1]:
from k2lake import connect, pin, SCALE
con = connect()
PIN = pin(con)   # every query below reads these snapshot ids, never the moving head
DAY = con.sql("SELECT max(exchange_ts)::DATE FROM pinned.gold_trades").fetchone()[0]
print('day', DAY)

pinned 18 tables at commit b12cff9
  audit.checks                 4558366752866520299
  gold.bars                    1589136844755948459
  gold.bbo_1s                  1578478823608857678
  gold.book_state              6881440282983849882
  gold.book_top20              1206037850593976770
  gold.dim_instrument          1768279322099994693
  gold.dim_venue               4191380317728821135
  gold.ohlcv_1d                1579946688158752964
  gold.ohlcv_1h                2397688695969945394
  gold.ohlcv_1m                1622213366608023449
  gold.ohlcv_5m                1666460790140246036
  gold.trades                  1348216816157062508
  silver.book_binance          7922581822436893978
  silver.book_coinbase         5196050954894723418
  silver.book_kraken           9210612890945089666
  silver.trades_binance        3528392281922118482
  silver.trades_coinbase       7144589879523206615
  silver.trades_kraken         264358488000747777
day 2026-08-27


In [2]:
con.sql(f"""
CREATE OR REPLACE TEMP TABLE t AS
SELECT exchange, canonical_symbol, exchange_ts, side, price_e8 / 100000000 AS price, qty_e8 / 100000000 AS qty
FROM pinned.gold_trades WHERE exchange_ts::DATE = DATE '{DAY}' AND canonical_symbol IN ('BTC/USD', 'BTC/USDT');
CREATE OR REPLACE TEMP TABLE q AS
SELECT exchange, canonical_symbol, second, bid_e8 / 100000000 AS bid, ask_e8 / 100000000 AS ask, mid
FROM pinned.gold_bbo_1s WHERE second::DATE = DATE '{DAY}' AND canonical_symbol IN ('BTC/USD', 'BTC/USDT');
""")
con.sql("SELECT (SELECT count(*) FROM t) AS trades, (SELECT count(*) FROM q) AS quotes").show()

┌────────┬────────┐
│ trades │ quotes │
│ int64  │ int64  │
├────────┼────────┤
│ 796782 │  56699 │
└────────┴────────┘



In [3]:
con.sql("""
CREATE OR REPLACE TEMP TABLE tq AS
SELECT t.*, q.bid, q.ask, q.mid, q.second AS quote_second
FROM t ASOF JOIN q ON t.exchange = q.exchange AND t.canonical_symbol = q.canonical_symbol AND t.exchange_ts >= q.second + INTERVAL 1 SECOND;
SELECT exchange, count(*) AS trades,
       round(100.0 * avg(CASE WHEN price BETWEEN bid AND ask THEN 1 ELSE 0 END), 2) AS pct_inside_or_at,
       round(100.0 * avg(CASE WHEN price > ask OR price < bid THEN 1 ELSE 0 END), 2) AS pct_through,
       round(avg(2 * abs(price - mid) / mid * 1e4), 3) AS effective_spread_bps,
       round(avg(CASE WHEN side = 'buy' THEN price - mid ELSE mid - price END) / avg(mid) * 1e4, 3) AS signed_cost_bps
FROM tq GROUP BY exchange ORDER BY exchange
""").show()

┌──────────┬────────┬──────────────────┬─────────────┬──────────────────────┬─────────────────┐
│ exchange │ trades │ pct_inside_or_at │ pct_through │ effective_spread_bps │ signed_cost_bps │
│ varchar  │ int64  │      double      │   double    │        double        │     double      │
├──────────┼────────┼──────────────────┼─────────────┼──────────────────────┼─────────────────┤
│ binance  │ 662376 │            32.55 │       67.45 │                 0.81 │           0.364 │
│ coinbase │ 121475 │             65.4 │        34.6 │                0.489 │          -0.102 │
│ kraken   │  12793 │            69.44 │       30.56 │                0.593 │           0.263 │
└──────────┴────────┴──────────────────┴─────────────┴──────────────────────┴─────────────────┘



A trade `through` the quote is either a real sweep or a quote that was already stale: the BBO is between one and two seconds old by construction (end of the previous second). The split by age says how much of the through-rate is staleness.

In [4]:
con.sql("""
SELECT exchange, CASE WHEN exchange_ts - quote_second < INTERVAL '1500' MILLISECOND THEN '1.0-1.5 s' ELSE '1.5-2 s+' END AS quote_age,
       count(*) AS trades, round(100.0 * avg(CASE WHEN price > ask OR price < bid THEN 1 ELSE 0 END), 2) AS pct_through
FROM tq GROUP BY 1, 2 ORDER BY 1, 2
""").show()

┌──────────┬───────────┬────────┬─────────────┐
│ exchange │ quote_age │ trades │ pct_through │
│ varchar  │  varchar  │ int64  │   double    │
├──────────┼───────────┼────────┼─────────────┤
│ binance  │ 1.0-1.5 s │ 322202 │       66.46 │
│ binance  │ 1.5-2 s+  │ 340174 │       68.39 │
│ coinbase │ 1.0-1.5 s │  61276 │       29.75 │
│ coinbase │ 1.5-2 s+  │  60199 │       39.54 │
│ kraken   │ 1.0-1.5 s │   7028 │       25.71 │
│ kraken   │ 1.5-2 s+  │   5765 │       36.46 │
└──────────┴───────────┴────────┴─────────────┘



## The security master, as of the trade

`gold.dim_instrument` is SCD2 ([ADR-030](../docs/adr/ADR-030-scd2-security-master.md)): one row per validity interval, so the instrument's attributes are read *as they stood when the trade printed*, not as they stand now. Same `ASOF JOIN` shape as the book above — the greatest `valid_from` at or before `exchange_ts`.

Two things to know before quoting anything from it. `tick_size` is populated for Kraken only (`source = 'venue:kraken'`, from `bronze.kraken_instrument`); Binance and Coinbase publish theirs over REST that K2 does not capture, so those are `NULL` and `source` says `registry`. And a trade older than the dimension's first version matches nothing: SCD2 landed on 2026-08-29 and there is no source anywhere for what the registry said before that.

The hand-written range join below is the same question spelled without `ASOF`, and it returns the same count *because* open rows carry `valid_to = 9999-12-31` rather than `NULL` — with `NULL`, `exchange_ts < valid_to` is `NULL`, not `TRUE`, and every current version would silently vanish from the result.

In [ ]:
con.sql("""
SELECT t.exchange, t.canonical_symbol, count(*) AS trades,
       any_value(d.symbol) AS native, any_value(d.book_depth) AS depth,
       any_value(d.tick_size) AS tick_size, any_value(d.source) AS source,
       any_value(d.valid_from) AS attrs_from
FROM t ASOF JOIN pinned.gold_dim_instrument d
  ON t.exchange = d.exchange AND t.canonical_symbol = d.canonical_symbol
 AND t.exchange_ts >= d.valid_from
GROUP BY 1, 2 ORDER BY 1, 2
""").show()

In [ ]:
# The same join without ASOF. Equal counts is the sentinel doing its job.
con.sql("""
SELECT (SELECT count(*) FROM t ASOF JOIN pinned.gold_dim_instrument d
          ON t.exchange = d.exchange AND t.canonical_symbol = d.canonical_symbol
         AND t.exchange_ts >= d.valid_from)                                  AS asof_rows,
       (SELECT count(*) FROM t JOIN pinned.gold_dim_instrument d
          ON t.exchange = d.exchange AND t.canonical_symbol = d.canonical_symbol
         AND t.exchange_ts >= d.valid_from AND t.exchange_ts < d.valid_to)   AS range_rows
""").show()